# Amostragem: GPT-5.4 `function_calling` vs `json_schema`

Este notebook verifica se a discrepância de resultados entre o `ner.ipynb` e o
`ner_openrouter_multi_model.ipynb` se deve à diferença no método de structured
output (`function_calling` vs padrão/`json_schema`).

**Estratégia**: rodar o GPT-5.4 em uma amostra estratificada de ~100 documentos
usando `method="function_calling"` e comparar com os resultados já existentes
(que usaram o método padrão).

**Variáveis controladas**:
- Mesmos exemplos few-shot (`generate_few_shot_ner_prompts`)
- Mesmo schema Pydantic (`NERDecisao`)
- Mesmos documentos (interseção por índice)
- Mesmo retry com fallback (sem substituir erros por vazio)

**Variável independente**: `method` do `with_structured_output`

In [1]:
import json
import os
import time

import numpy as np
import pandas as pd
import spacy

from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
from rapidfuzz import fuzz
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, AzureChatOpenAI
from sklearn.metrics import precision_recall_fscore_support

from tools.dataset import get_decicontas_df
from tools.prompt import generate_few_shot_ner_prompts
from tools.schema import NERDecisao

load_dotenv()

SAMPLE_SIZE = 100
SEED = 42
MAX_RETRIES = 5
OUTPUT_DIR = Path("dataset/results/sampling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregar dataset e resultados existentes

In [2]:
df_decicontas = get_decicontas_df()
print(f"Dataset total: {len(df_decicontas)} documentos")

# Carregar resultados existentes do gpt-5.4 few_shot (método padrão/json_schema)
EXISTING_RESULTS_PATH = "dataset/results/models_results_decicontas_gpt-5.4_few_shot.json"
df_existing = pd.read_json(EXISTING_RESULTS_PATH)
print(f"Resultados existentes (gpt-5.4 few_shot): {len(df_existing)} documentos")

Dataset total: 866 documentos
Resultados existentes (gpt-5.4 few_shot): 866 documentos


## 2. Amostragem estratificada

Garante representação proporcional de documentos com e sem cada tipo de entidade.

In [3]:
# Índices disponíveis nos resultados existentes
available_indices = set(df_existing['index'].tolist())

# Criar flags de presença de cada entidade
entity_flags = []
for idx, row in df_decicontas.iterrows():
    if idx not in available_indices:
        continue
    annotations = row['annotations'][0]['result']
    labels = set(r['value']['labels'][0] for r in annotations) if annotations else set()
    entity_flags.append({
        'index': idx,
        'has_multa': 'MULTA' in labels,
        'has_obrigacao': 'OBRIGACAO' in labels,
        'has_recomendacao': 'RECOMENDACAO' in labels,
        'has_ressarcimento': 'RESSARCIMENTO' in labels,
        'has_any': len(labels) > 0,
    })

df_flags = pd.DataFrame(entity_flags)

# Estratificar: combinar flags em um grupo
df_flags['stratum'] = (
    df_flags['has_multa'].astype(int).astype(str) + '_' +
    df_flags['has_obrigacao'].astype(int).astype(str) + '_' +
    df_flags['has_recomendacao'].astype(int).astype(str) + '_' +
    df_flags['has_ressarcimento'].astype(int).astype(str)
)

print("Distribuição dos estratos:")
print(df_flags['stratum'].value_counts())
print(f"\nTotal disponível: {len(df_flags)}")

# Amostragem proporcional
np.random.seed(SEED)
sample_indices = (
    df_flags
    .groupby('stratum', group_keys=False)
    .apply(lambda x: x.sample(frac=SAMPLE_SIZE / len(df_flags), random_state=SEED))
)

# Ajustar para exatamente SAMPLE_SIZE
if len(sample_indices) < SAMPLE_SIZE:
    remaining = df_flags[~df_flags.index.isin(sample_indices.index)]
    extra = remaining.sample(n=SAMPLE_SIZE - len(sample_indices), random_state=SEED)
    sample_indices = pd.concat([sample_indices, extra])
elif len(sample_indices) > SAMPLE_SIZE:
    sample_indices = sample_indices.sample(n=SAMPLE_SIZE, random_state=SEED)

SAMPLE_IDS = sorted(sample_indices['index'].tolist())
print(f"\nAmostra final: {len(SAMPLE_IDS)} documentos")
print(f"Com entidades: {sample_indices['has_any'].sum()}")
print(f"Sem entidades: {(~sample_indices['has_any']).sum()}")

Distribuição dos estratos:
stratum
0_0_0_0    637
1_1_0_0     55
1_0_0_0     46
0_0_1_0     41
1_0_0_1     34
0_1_0_0     23
0_0_0_1     18
0_1_1_0      7
1_0_1_1      2
1_1_1_0      1
0_1_0_1      1
1_1_0_1      1
Name: count, dtype: int64

Total disponível: 866

Amostra final: 100 documentos
Com entidades: 26
Sem entidades: 74


/var/folders/39/p0t9g2qn1gbcvnwqyz79d34c0000gn/T/ipykernel_87163/3643439334.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=SAMPLE_SIZE / len(df_flags), random_state=SEED))


## 3. Configurar extratores

Dois extratores idênticos, diferindo apenas no `method`.

In [4]:
# ============================================================
# AJUSTE AQUI: usar Azure ou OpenRouter conforme seu setup
# ============================================================

# Opção A: Azure (descomente se gpt-5.4 estiver na Azure)
base_llm = AzureChatOpenAI(
     azure_deployment="gpt-5.4",
     azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
     api_key=os.getenv("AZURE_OPENAI_API_KEY"),
     api_version=os.getenv("OPENAI_API_VERSION"),
 )

# Opção B: OpenRouter (como no ner_openrouter_multi_model.ipynb)
"""
base_llm = ChatOpenAI(
    model="openai/gpt-5.4",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "https://github.com/decicontas",
        "X-Title": "decicontas-ner",
    },
    max_tokens=4096,
)
"""

# Extrator A: function_calling (mesmo método do ner.ipynb)
extractor_fc = base_llm.with_structured_output(
    NERDecisao, include_raw=False, method="function_calling"
)

# Extrator B: padrão (mesmo método do ner_openrouter_multi_model.ipynb)
extractor_default = base_llm.with_structured_output(
    NERDecisao, include_raw=False
)

print("Extratores configurados:")
print(f"  A) function_calling  — replica ner.ipynb")
print(f"  B) default (json_schema) — replica ner_openrouter_multi_model.ipynb")

Extratores configurados:
  A) function_calling  — replica ner.ipynb
  B) default (json_schema) — replica ner_openrouter_multi_model.ipynb


## 4. Executar extração na amostra

In [5]:
def run_extraction(extractor, method_name, sample_ids, df_data):
    """Roda extração com retry (sem substituir erros por vazio)."""
    results = []
    errors = 0

    for doc_idx in tqdm(sample_ids, desc=method_name, unit="doc"):
        row = df_data.loc[doc_idx]
        text = row['data']['text']
        golden = [r['value'] for r in row['annotations'][0]['result']]
        prompt = generate_few_shot_ner_prompts(text)

        success = False
        retries = 0
        while not success and retries < MAX_RETRIES:
            try:
                result = extractor.invoke(prompt)
                success = True
            except Exception as e:
                retries += 1
                print(f"  Erro doc {doc_idx} (tentativa {retries}): {e}")
                time.sleep(3)

        if not success:
            errors += 1
            result = NERDecisao(
                multas=[], obrigacoes=[], recomendacoes=[], ressarcimentos=[]
            )
            print(f"  FALHA DEFINITIVA doc {doc_idx} após {MAX_RETRIES} tentativas")

        results.append({
            'index': doc_idx,
            'text': text,
            'pred': result.model_dump(),
            'golden': golden,
            'model': 'gpt-5.4',
            'technique': f'few_shot_{method_name}',
        })

    print(f"\n{method_name}: {len(results)} docs processados, {errors} erros")
    return pd.DataFrame(results)

In [6]:
# Verificar se já existe resultado salvo
fc_path = OUTPUT_DIR / "gpt54_sample_function_calling.json"

if fc_path.exists():
    print(f"Resultados function_calling já existem: {fc_path}")
    df_fc = pd.read_json(fc_path)
else:
    print("Rodando extração com function_calling...")
    df_fc = run_extraction(extractor_fc, "function_calling", SAMPLE_IDS, df_decicontas)
    df_fc.to_json(fc_path, orient="records", force_ascii=False, indent=2)
    print(f"Salvo em {fc_path}")

Rodando extração com function_calling...


function_calling: 100%|██████████| 100/100 [02:45<00:00,  1.66s/doc]


function_calling: 100 docs processados, 0 erros
Salvo em dataset/results/sampling/gpt54_sample_function_calling.json


In [7]:
# Extrair resultados existentes (default) para os mesmos índices da amostra
df_default = df_existing[df_existing['index'].isin(SAMPLE_IDS)].copy()
df_default['technique'] = 'few_shot_default'
print(f"Resultados default filtrados: {len(df_default)} docs")

if len(df_default) < len(SAMPLE_IDS):
    missing = set(SAMPLE_IDS) - set(df_default['index'].tolist())
    print(f"  AVISO: {len(missing)} docs da amostra não encontrados nos resultados existentes")
    SAMPLE_IDS_FINAL = sorted(set(SAMPLE_IDS) & set(df_default['index'].tolist()))
    df_fc = df_fc[df_fc['index'].isin(SAMPLE_IDS_FINAL)]
    df_default = df_default[df_default['index'].isin(SAMPLE_IDS_FINAL)]
    print(f"  Amostra ajustada: {len(SAMPLE_IDS_FINAL)} docs")

Resultados default filtrados: 100 docs


## 5. Avaliação (mesma pipeline do notebook original)

In [8]:
DICT_LABELS = {
    "obrigacoes": "OBRIGACAO",
    "recomendacoes": "RECOMENDACAO",
    "ressarcimentos": "RESSARCIMENTO",
    "multas": "MULTA",
}

def convert_pred_to_golden_format(row, window_size=500, step_size=100, min_score=80):
    pred_spans = []
    text = row['text']
    pred = row['pred']
    for label_type, spans in pred.items():
        for span in (spans or []):
            if not isinstance(span, dict):
                continue
            span_text = (
                span.get("descricao_multa") or span.get("descricao_obrigacao")
                or span.get("descricao_ressarcimento") or span.get("descricao_recomendacao")
            )
            if not span_text:
                continue
            best_score, best_pos, best_substring = 0, -1, ""
            for start in range(0, len(text), step_size):
                window = text[start:start + window_size]
                score = fuzz.partial_ratio(span_text, window)
                if score > best_score and score >= min_score:
                    best_score = score
                    best_pos = start + window.find(span_text.split()[0]) if span_text.split() else start
                    best_substring = span_text
            if best_score >= min_score and best_pos >= 0:
                pred_spans.append({
                    "start": best_pos, "end": best_pos + len(best_substring),
                    "text": best_substring, "labels": [DICT_LABELS[label_type]]
                })
    return pred_spans


def _strip_bio(tag):
    if tag == 'O' or tag is None:
        return 'O'
    parts = tag.split('-', 1)
    return parts[1] if len(parts) == 2 else parts[0]


def compute_iou_score(span_a, span_b, label_a, label_b, threshold=0.5):
    s_a, e_a = span_a
    s_b, e_b = span_b
    if e_a <= s_b or e_b <= s_a:
        return 0.0
    intersection = max(0, min(e_a, e_b) - max(s_a, s_b))
    union = max(e_a, e_b) - min(s_a, s_b)
    iou = intersection / union if union > 0 else 0.0
    return 1.0 if (iou >= threshold and label_a == label_b) else 0.0


def calculate_metrics(df, iou_threshold=0.5, spacy_model="pt_core_news_sm"):
    nlp = spacy.load(spacy_model)
    y_true_tokens, y_pred_tokens = [], []
    label_metrics = defaultdict(lambda: {"total_gold": 0, "total_pred": 0, "matched": 0})

    for _, row in df.iterrows():
        text = row["text"]
        doc = nlp(text)
        true_bio = ['O'] * len(doc)
        pred_bio = ['O'] * len(doc)

        for ann in row.get("golden", []):
            start, end, label = ann["start"], ann["end"], ann["labels"][0]
            cs = doc.char_span(start, end, label=label, alignment_mode="expand")
            if cs:
                for j, tok in enumerate(cs):
                    true_bio[tok.i] = f"B-{label}" if j == 0 else f"I-{label}"

        for ann in row.get("pred_as_golden", []):
            start, end, label = ann["start"], ann["end"], ann["labels"][0]
            cs = doc.char_span(start, end, label=label, alignment_mode="expand")
            if cs:
                for j, tok in enumerate(cs):
                    pred_bio[tok.i] = f"B-{label}" if j == 0 else f"I-{label}"

        y_true_tokens.append([_strip_bio(t) for t in true_bio])
        y_pred_tokens.append([_strip_bio(t) for t in pred_bio])

        gold_spans = [(a['start'], a['end'], a['labels'][0]) for a in row.get('golden', [])]
        pred_spans = [(a['start'], a['end'], a['labels'][0]) for a in row.get('pred_as_golden', [])]
        for _, _, lab in gold_spans:
            label_metrics[lab]["total_gold"] += 1
        for _, _, lab in pred_spans:
            label_metrics[lab]["total_pred"] += 1

        matched_pairs = set()
        for pi, p in enumerate(pred_spans):
            for gi, g in enumerate(gold_spans):
                if (pi, gi) in matched_pairs:
                    continue
                if compute_iou_score((p[0], p[1]), (g[0], g[1]), p[2], g[2], threshold=iou_threshold) > 0:
                    label_metrics[p[2]]["matched"] += 1
                    matched_pairs.add((pi, gi))
                    break

    # Token-level (flat, sem O, sem B/I)
    all_true = [t for seq in y_true_tokens for t in seq if t != 'O']
    all_pred = [p for seq_t, seq_p in zip(y_true_tokens, y_pred_tokens) for t, p in zip(seq_t, seq_p) if t != 'O']
    # Alinhar: pegar todos os tokens onde true OU pred != O
    all_true_full, all_pred_full = [], []
    for seq_t, seq_p in zip(y_true_tokens, y_pred_tokens):
        for t, p in zip(seq_t, seq_p):
            if t != 'O' or p != 'O':
                all_true_full.append(t)
                all_pred_full.append(p)

    labels = sorted(set(all_true_full + all_pred_full) - {'O'})
    p, r, f1, _ = precision_recall_fscore_support(
        all_true_full, all_pred_full, labels=labels, average='weighted', zero_division=0
    )

    # Span-level IoU
    total_matched = sum(v["matched"] for v in label_metrics.values())
    total_pred = sum(v["total_pred"] for v in label_metrics.values())
    total_gold = sum(v["total_gold"] for v in label_metrics.values())
    span_p = total_matched / total_pred if total_pred > 0 else 0
    span_r = total_matched / total_gold if total_gold > 0 else 0
    span_f1 = 2 * span_p * span_r / (span_p + span_r) if (span_p + span_r) > 0 else 0

    # Per-label IoU
    iou_per_label = {}
    for lab, vals in label_metrics.items():
        lp = vals["matched"] / vals["total_pred"] if vals["total_pred"] > 0 else 0
        lr = vals["matched"] / vals["total_gold"] if vals["total_gold"] > 0 else 0
        lf = 2 * lp * lr / (lp + lr) if (lp + lr) > 0 else 0
        iou_per_label[lab] = {"precision": lp, "recall": lr, "f1": lf,
                              "gold": vals["total_gold"], "pred": vals["total_pred"],
                              "matched": vals["matched"]}

    return {
        "token_flat": {"precision": p, "recall": r, "f1": f1},
        "iou_agg": {"precision": span_p, "recall": span_r, "f1": span_f1},
        "iou_per_label": iou_per_label,
    }

In [9]:
# Preparar predições no formato golden
df_fc['pred_as_golden'] = df_fc.apply(convert_pred_to_golden_format, axis=1)
df_default['pred_as_golden'] = df_default.apply(convert_pred_to_golden_format, axis=1)

print("Calculando métricas...")
metrics_fc = calculate_metrics(df_fc)
metrics_default = calculate_metrics(df_default)
print("Pronto!")

Calculando métricas...
Pronto!


## 6. Comparação dos resultados

In [10]:
def format_metrics_row(name, m):
    return {
        "Método": name,
        "Token P": round(m["token_flat"]["precision"], 4),
        "Token R": round(m["token_flat"]["recall"], 4),
        "Token F1": round(m["token_flat"]["f1"], 4),
        "Span P": round(m["iou_agg"]["precision"], 4),
        "Span R": round(m["iou_agg"]["recall"], 4),
        "Span F1": round(m["iou_agg"]["f1"], 4),
    }

df_comparison = pd.DataFrame([
    format_metrics_row("function_calling", metrics_fc),
    format_metrics_row("default (json_schema)", metrics_default),
])

print("=" * 70)
print("COMPARAÇÃO: function_calling vs default (mesma amostra)")
print("=" * 70)
print(df_comparison.to_string(index=False))

# Delta
delta_token = metrics_fc["token_flat"]["f1"] - metrics_default["token_flat"]["f1"]
delta_span = metrics_fc["iou_agg"]["f1"] - metrics_default["iou_agg"]["f1"]
print(f"\nDelta Token F1: {delta_token:+.4f}")
print(f"Delta Span F1:  {delta_span:+.4f}")

if abs(delta_span) > 0.02:
    print("\n→ DIFERENÇA SIGNIFICATIVA: o método de structured output impacta os resultados.")
    print("  A comparação entre notebooks NÃO é direta sem controlar essa variável.")
else:
    print("\n→ Diferença pequena: o método de structured output NÃO explica a discrepância.")
    print("  A diferença entre gpt-5.4 e gpt-4-turbo é provavelmente real.")

COMPARAÇÃO: function_calling vs default (mesma amostra)
               Método  Token P  Token R  Token F1  Span P  Span R  Span F1
     function_calling   0.8340   0.8550    0.8395  0.7500   0.900   0.8182
default (json_schema)   0.8543   0.8279    0.8311  0.6863   0.875   0.7692

Delta Token F1: +0.0084
Delta Span F1:  +0.0490

→ DIFERENÇA SIGNIFICATIVA: o método de structured output impacta os resultados.
  A comparação entre notebooks NÃO é direta sem controlar essa variável.


In [11]:
# Per-label comparison
print("\n" + "=" * 70)
print("COMPARAÇÃO POR ENTIDADE (Span-level IoU)")
print("=" * 70)

all_labels = sorted(
    set(list(metrics_fc["iou_per_label"].keys()) + list(metrics_default["iou_per_label"].keys()))
)

rows = []
for label in all_labels:
    fc_vals = metrics_fc["iou_per_label"].get(label, {"precision": 0, "recall": 0, "f1": 0})
    def_vals = metrics_default["iou_per_label"].get(label, {"precision": 0, "recall": 0, "f1": 0})
    rows.append({
        "Entidade": label,
        "FC Prec": round(fc_vals["precision"], 4),
        "FC Rec": round(fc_vals["recall"], 4),
        "FC F1": round(fc_vals["f1"], 4),
        "Def Prec": round(def_vals["precision"], 4),
        "Def Rec": round(def_vals["recall"], 4),
        "Def F1": round(def_vals["f1"], 4),
        "Delta F1": round(fc_vals["f1"] - def_vals["f1"], 4),
    })

df_per_label = pd.DataFrame(rows)
print(df_per_label.to_string(index=False))


COMPARAÇÃO POR ENTIDADE (Span-level IoU)
     Entidade  FC Prec  FC Rec  FC F1  Def Prec  Def Rec  Def F1  Delta F1
        MULTA   0.9444  1.0000 0.9714    0.8947   1.0000  0.9444    0.0270
    OBRIGACAO   0.8333  0.9091 0.8696    0.7143   0.9091  0.8000    0.0696
 RECOMENDACAO   0.5556  0.8333 0.6667    0.4444   0.6667  0.5333    0.1333
RESSARCIMENTO   0.4444  0.6667 0.5333    0.4444   0.6667  0.5333    0.0000


In [12]:
# Contar erros de parsing por método
fc_empty = df_fc.apply(
    lambda r: all(len(r['pred'].get(k, []) or []) == 0 for k in DICT_LABELS), axis=1
).sum()
def_empty = df_default.apply(
    lambda r: all(len(r['pred'].get(k, []) or []) == 0 for k in DICT_LABELS), axis=1
).sum()

# Contar docs com golden vazio para referência
gold_empty = df_fc.apply(lambda r: len(r['golden']) == 0, axis=1).sum()

print(f"\nPredições completamente vazias:")
print(f"  function_calling: {fc_empty}/{len(df_fc)} ({fc_empty/len(df_fc)*100:.1f}%)")
print(f"  default:          {def_empty}/{len(df_default)} ({def_empty/len(df_default)*100:.1f}%)")
print(f"  golden vazio:     {gold_empty}/{len(df_fc)} ({gold_empty/len(df_fc)*100:.1f}%)")


Predições completamente vazias:
  function_calling: 69/100 (69.0%)
  default:          69/100 (69.0%)
  golden vazio:     74/100 (74.0%)


## 7. Conclusão

Interpretação dos resultados:

- **Delta Span F1 > +0.02**: O `function_calling` melhora significativamente os
  resultados. A diferença observada entre gpt-5.4 e gpt-4-turbo no artigo é
  parcialmente (ou totalmente) explicada pelo método de structured output. Para
  comparação justa, re-rodar o dataset completo com `function_calling`.

- **Delta Span F1 entre -0.02 e +0.02**: O método de structured output não é o
  fator. A diferença entre os modelos é real e pode ser atribuída às capacidades
  intrínsecas dos modelos.

- **Delta Span F1 < -0.02**: O `function_calling` piora os resultados (improvável,
  mas possível se o modelo foi otimizado para json_schema).